In [0]:
# Databricks notebook source
from pyspark.sql.functions import lit
# ==========================================
# 1️⃣ Create Widget
# ==========================================

dbutils.widgets.text("CustomerCode_Source", "")
CustomerCode_Source = dbutils.widgets.get("CustomerCode_Source")

dbutils.widgets.text(
    "run_id",
    "472519629468310"
)
run_id = dbutils.widgets.get("run_id")

print(f"Source System Passed: {CustomerCode_Source}")

if not CustomerCode_Source:
    raise ValueError("CustomerCode_Source widget cannot be empty")


# ==========================================
# 2️⃣ Read Metadata Tables
# ==========================================

tables_df = spark.sql("""select TL.CustomerProductID,TL.TableID,C.CustomerName,C.CustomerCode,TL.SourceSchema,TL.SourceTablename, TL.ExtractionType, TL.WatermarkColumn, TL.LastExtractWatermark,CD.ConnectionName, CD.SecretKeyName,TL.IsActive from clinicalforge.metadata.TablesList as TL
left join clinicalforge.metadata.ConnectionDetails as CD on TL.SourceConnectionID=CD.ConnectionID and TL.CustomerProductID=CD.CustomerProductID
left Join clinicalforge.metadata.CustomerProduct as CP on CP.CustomerProductID=TL.CustomerProductID
left join clinicalforge.metadata.customers as C on CP.CustomerID=C.CustomerID             
""")

# where C.CustomerCode= '{CustomerCode_Source}'  
filtered_df = (
    tables_df
    .filter(f"lower(CustomerCode)=lower('{CustomerCode_Source}') and IsActive=true")
    .withColumn('TargetSchema',lit('Bronze'))
    .orderBy("TableID")
)

display(filtered_df)


# ==========================================
# 3️⃣ Convert to List of Dictionaries
# ==========================================

rows = filtered_df.collect()

tables_list = [
    {
        "CustomerProductID": str(row.CustomerProductID) if row.CustomerProductID is not None else "",
        "TableID": str(row.TableID) if row.TableID is not None else "",
        "SourceTablename": str(row.SourceTablename) if row.SourceTablename is not None else "",
        "SourceSchema": str(row.SourceSchema) if row.SourceSchema is not None else "",
        "CustomerName": str(row.CustomerName) if row.CustomerName is not None else "",
        "CustomerCode": str(row.CustomerCode) if row.CustomerCode is not None else "",
        "ExtractionType": str(row.ExtractionType) if row.ExtractionType is not None else "",
        "WatermarkColumn": str(row.WatermarkColumn) if row.WatermarkColumn is not None else "",
        "LastExtractWatermark": str(row.LastExtractWatermark) if row.LastExtractWatermark is not None else "",
        "TargetSchema": str(row.TargetSchema) if row.TargetSchema is not None else "",
        "IsActive": str(row.IsActive) if row.IsActive is not None else ""
    }
    for row in rows
]

print("Tables to Process (Full Metadata):")
print(tables_list)


# ==========================================
# 4️⃣ Set Databricks Task Value
# ==========================================

dbutils.jobs.taskValues.set(
    key="tables_metadata",
    value=tables_list
)

print("Task value 'tables_metadata' has been set.")